In [ ]:
!pip install yfinance
!pip install ta

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=2234a3aebbda19fc43c2cea60efff369e9d8eefe0ec2a26f231522f0366de864
  Stored in directory: /root/.cache/pip/wheels/5f/67/4f/8a9f252836e053e532c6587a3230bc72a4deb16b03a829610b
Successfully built ta


In [ ]:
import pandas as pd
import yfinance as yf
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Data Retrieval

Let's obtain all NasDaq30 data

In [ ]:
nasdaq_30 = [
    'AAPL',  # Apple Inc.
    'MSFT',  # Microsoft Corporation
    'AMZN',  # Amazon.com, Inc.
    'NVDA',  # NVIDIA Corporation
    'GOOGL', # Alphabet Inc.
    'META',  # Meta Platforms, Inc.
    'TSLA',  # Tesla, Inc.
    'AVGO',  # Broadcom Inc.
    'PEP',   # PepsiCo, Inc.
    'COST',  # Costco Wholesale Corporation
    'CSCO',  # Cisco Systems, Inc.
    'ADBE',  # Adobe Inc.
    'NFLX',  # Netflix, Inc.
    'AMD',   # Advanced Micro Devices, Inc.
    'GOOG',  # Alphabet Inc.
    'TXN',   # Texas Instruments Incorporated
    'INTC',  # Intel Corporation
    'AMGN',  # Amgen Inc.
    'HON',   # Honeywell International Inc.
    'CMCSA', # Comcast Corporation
    'ADP',   # Automatic Data Processing, Inc.
    'PYPL',  # PayPal Holdings, Inc.
    'QCOM',  # Qualcomm Incorporated
    'ISRG',  # Intuitive Surgical, Inc.
    'CHTR',  # Charter Communications, Inc.
    'SBUX',  # Starbucks Corporation
    'GILD',  # Gilead Sciences, Inc.
    'MDLZ',  # Mondelez International, Inc.
    'ADI',   # Analog Devices, Inc.
    'BKNG'   # Booking Holdings Inc.
]


In [ ]:
def hist_data(symbols):

  # Initialize an empty list to store DataFrames for each symbol
  dfs = {}

  for i in range(1, len(symbols)+1):

      print(symbols[i-1])

      '''

      url = f'https://eodhd.com/api/eod/{symbols[i-1]}.NSE?from={ten_years_ago}&to={today}&api_token=5ffde981a31603.50022638&fmt=json' # with free API token, I can only do over the past year
      nasdaq = requests.get(url).json()
      '''

      stock = yf.Ticker(symbols[i-1])
      historical_data = stock.history(period='10y')  # Data for the past 10 years

      # Convert data to DataFrame
      dfs[f'df{i}'] = pd.DataFrame(historical_data)

      # changing the date to the index column
      #dfs[f'df{i}'] = dfs[f'df{i}'].set_index('date')

      # renaming the 'close' column to the name of the symbol
      #dfs[f'df{i}'] = dfs[f'df{i}'].rename(columns={'close': symbols[i-1]})

      # adding returns
      dfs[f'df{i}']['Returns'] = (dfs[f'df{i}']['Close'] / dfs[f'df{i}']['Close'].shift(1)) - 1

      # dropping rows with NaN values
      dfs[f'df{i}'] = dfs[f'df{i}'].dropna()

      # using only specific coloumns
      dfs[f'df{i}'] = dfs[f'df{i}'][['Open', 'High', 'Low', 'Close', 'Volume', 'Returns']] # not adding returns here

      print(dfs[f'df{i}'])

  return dfs


In [ ]:
def get_returns(symbols):

    # Dictionary to hold returns for each company
    returns_dict = {}

    for i in range(1, len(symbols)+1):

      '''
      url = f'https://eodhd.com/api/eod/{symbols[i-1]}.NSE?from={start}&to={end}&api_token=5ffde981a31603.50022638&fmt=json' # with free API token, I can only do over the past year
      nasdaq = requests.get(url).json()
      '''

      stock = yf.Ticker(symbols[i-1])
      historical_data = stock.history(period='10y')  # Data for the past 10 years

      # adding returns
      historical_data['Returns'] = (historical_data['Close']/ historical_data['Close'].shift(1)) - 1

      # drop NAN
      historical_data = historical_data.dropna()

      returns_dict[symbols[i-1]] = historical_data['Returns']

    # Create a DataFrame from the returns dictionary
    returns_df = pd.DataFrame(returns_dict)

    # Drop the first row with NaN values due to pct_change
    returns_df = returns_df.dropna()

    return returns_df


In [ ]:
#gathering everything
data_dfs = hist_data(nasdaq_30)
return_data = get_returns(nasdaq_30)

AAPL
                                 Open        High         Low       Close  \
Date                                                                        
2014-11-11 00:00:00-05:00   24.206875   24.440705   24.140068   24.429569   
2014-11-12 00:00:00-05:00   24.358305   24.814829   24.356079   24.774744   
2014-11-13 00:00:00-05:00   24.897227   25.264671   24.852687   25.124374   
2014-11-14 00:00:00-05:00   25.197858   25.429460   24.765830   25.427233   
2014-11-17 00:00:00-05:00   25.447282   26.117592   25.231270   25.384928   
...                               ...         ...         ...         ...   
2024-11-04 00:00:00-05:00  220.747138  222.545148  219.468546  221.766006   
2024-11-05 00:00:00-05:00  221.556241  223.703872  220.896963  223.204422   
2024-11-06 00:00:00-05:00  222.365355  225.821559  220.946918  222.475235   
2024-11-07 00:00:00-05:00  224.383137  227.629565  224.323205  227.229996   
2024-11-08 00:00:00-05:00  227.169998  228.660004  226.410004  226.9600

In [ ]:
return_data

,AAPL,MSFT,AMZN,NVDA,GOOGL,META,TSLA,AVGO,PEP,COST,...,ADP,PYPL,QCOM,ISRG,CHTR,SBUX,GILD,MDLZ,ADI,BKNG
Date,,,,,,,,,,,,,,,,,,,,,
2015-07-07 00:00:00-04:00,-0.002461,-0.002028,0.001560,-0.018840,0.008083,-0.003769,-0.042328,-0.010134,0.021186,0.034000,...,0.002969,-0.002452,-0.004912,0.007049,0.004883,0.001289,-0.004583,0.012603,-0.003302,0.003136
2015-07-08 00:00:00-04:00,-0.024823,-0.001354,-0.016074,-0.007074,-0.015145,-0.018001,-0.048231,-0.036058,-0.008195,-0.004270,...,-0.012828,-0.052430,-0.014172,-0.005857,-0.015150,-0.018205,-0.019022,-0.008856,-0.022720,-0.024515
2015-07-09 00:00:00-04:00,-0.020397,0.006329,0.010915,-0.012214,0.005446,0.002685,0.011610,-0.026318,-0.010668,-0.003217,...,0.008247,-0.005764,-0.000808,0.005892,0.002206,0.012362,0.003719,-0.003864,-0.016951,0.008174
2015-07-10 00:00:00-04:00,0.026734,0.002021,0.020995,0.017517,0.021041,0.024103,0.004769,0.041386,0.010149,0.023450,...,0.010286,0.005507,0.014064,0.013293,0.027454,0.009621,0.003352,0.003879,0.010182,0.015203
2015-07-13 00:00:00-04:00,0.019306,0.020848,0.027192,0.006583,0.028088,0.024446,0.011615,-0.005083,0.017478,0.009389,...,0.002577,0.060248,0.011159,0.019821,0.006483,0.020708,0.007122,0.004588,0.005853,0.024668
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-04 00:00:00-05:00,-0.004038,-0.004654,-0.010862,0.004801,-0.011968,-0.011425,-0.024661,-0.002190,0.004529,0.009985,...,0.005830,0.011392,-0.000545,0.002133,0.025485,-0.018610,0.003687,-0.002189,-0.005721,0.005759
2024-11-05 00:00:00-05:00,0.006486,0.007345,0.019001,0.028372,0.002954,0.020957,0.035414,0.031741,0.009078,0.004627,...,0.004554,0.014335,0.004419,0.015273,0.017428,0.000618,0.004230,0.010094,-0.029395,0.029001
2024-11-06 00:00:00-05:00,-0.003267,0.021193,0.038045,0.040740,0.039884,-0.000664,0.147510,0.032490,-0.018707,0.010200,...,0.048080,0.027256,0.042674,0.000738,0.067054,-0.010403,0.016294,-0.038812,0.039752,0.017540


In [ ]:
'''

modifying the length of some of the dataframes
since PYPL data doesn't exist before 2015

'''

# Find the minimum and max number of rows across all DataFrames
min_length = min(df.shape[0] for df in data_dfs.values())
max_length = max(df.shape[0] for df in data_dfs.values())
print(min_length)

# Truncate all DataFrames to the minimum length
for key, df in data_dfs.items():
    data_dfs[key] = df.iloc[(max_length-min_length):]

print(data_dfs)

2354
{'df1':                                  Open        High         Low       Close  \
Date                                                                        
2015-07-07 00:00:00-04:00   28.263197   28.321569   27.787241   28.218296   
2015-07-08 00:00:00-04:00   27.946643   27.982563   27.511099   27.517834   
2015-07-09 00:00:00-04:00   27.805202   27.852349   26.765735   26.956566   
2015-07-10 00:00:00-04:00   27.376398   27.805206   27.212507   27.677237   
2015-07-13 00:00:00-04:00   28.070124   28.234015   27.910725   28.211565   
...                               ...         ...         ...         ...   
2024-11-04 00:00:00-05:00  220.747138  222.545148  219.468546  221.766006   
2024-11-05 00:00:00-05:00  221.556241  223.703872  220.896963  223.204422   
2024-11-06 00:00:00-05:00  222.365355  225.821559  220.946918  222.475235   
2024-11-07 00:00:00-05:00  224.383137  227.629565  224.323205  227.229996   
2024-11-08 00:00:00-05:00  227.169998  228.660004  226.410004  

# Normalise Pricing

In [ ]:
def normalise(df):

    # Create a copy to avoid modifying the original DataFrame
    df_normalized = df.copy()

    # Normalize each price column by dividing by the first value (t=1)
    for column in ['Open', 'High', 'Low', 'Close']:
        df_normalized[column] = df[column] / df[column].iloc[0]
        df_normalized.rename(columns={column: f"{column}_normalized"}, inplace=True)

    # Ensure the volume column is present without any changes
    df_normalized['Volume'] = df['Volume']

    # Display the modified DataFrame
    #print(df_normalized)

    return df_normalized


In [ ]:
def norm_dict(dict):

  norm_dict = {}

  for key, value in dict.items():
    norm_dict[key] = normalise(value)

  return norm_dict

In [ ]:
normalised_dict = norm_dict(data_dfs)
print(normalised_dict) # new line, get rid later

{'df1':                            Open_normalized  High_normalized  Low_normalized  \
Date                                                                          
2015-07-07 00:00:00-04:00         1.000000         1.000000        1.000000   
2015-07-08 00:00:00-04:00         0.988800         0.988030        0.990062   
2015-07-09 00:00:00-04:00         0.983795         0.983432        0.963238   
2015-07-10 00:00:00-04:00         0.968624         0.981768        0.979317   
2015-07-13 00:00:00-04:00         0.993169         0.996909        1.004444   
...                                    ...              ...             ...   
2024-11-04 00:00:00-05:00         7.810409         7.857797        7.898177   
2024-11-05 00:00:00-05:00         7.839037         7.898710        7.949582   
2024-11-06 00:00:00-05:00         7.867665         7.973483        7.951380   
2024-11-07 00:00:00-05:00         7.939057         8.037322        8.072885   
2024-11-08 00:00:00-05:00         8.037661  

# PCA

PC on returns

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
def pca_vectors(data):

    # Remove the date column
    #returned_data = data.drop(columns=['Date'])
    # should we perform PCA on returns also?

    # Perform PCA
    pca = PCA()
    pca.fit(data) #should we standardise the data ?

    principal_components = pca.components_

    return principal_components


In [ ]:
pc_vec = pca_vectors(return_data)

In [ ]:
pc_vec # check if this is correct

array([[ 1.77469603e-01,  1.79981838e-01,  1.90508875e-01,
         3.11430727e-01,  1.76800007e-01,  2.08716782e-01,
         2.78873078e-01,  2.18698476e-01,  7.19061170e-02,
         1.00726712e-01,  1.30593758e-01,  2.06400451e-01,
         2.13117062e-01,  3.20512519e-01,  1.76654790e-01,
         1.82173078e-01,  1.95349542e-01,  9.29603286e-02,
         1.12038639e-01,  1.10163631e-01,  1.25173767e-01,
         2.11930391e-01,  2.10963773e-01,  1.72273450e-01,
         1.18042998e-01,  1.35425465e-01,  7.60125346e-02,
         8.47737196e-02,  1.98189700e-01,  1.55301122e-01],
       [-7.10310553e-02, -1.08946919e-01, -5.97394936e-02,
         2.24305964e-01, -1.35154691e-01, -1.30090303e-01,
         3.82092483e-01,  7.99075341e-03, -1.30371059e-01,
        -7.95174187e-02, -1.27265476e-01, -8.24731734e-02,
        -5.91685317e-02,  6.77881051e-01, -1.35586513e-01,
        -3.78300424e-02, -8.68977579e-02, -1.46498089e-01,
        -1.35208794e-01, -1.64700074e-01, -1.49487052e-

# PC Portfolio

In [ ]:
def portfolio(pc_array, stock_data_list, comp): # comp defines which principal component we are looking at

    # Initialize an empty DataFrame to store the results
    combined_data = None

    # taking PC component as weight
    weights_array = (-1) * pc_array[:][comp-1]
    # print(weights_array)

    # now we want to take sum of each column according to the weight assigned to the stock

    # Initialize a dictionary to store the weighted sums for each column type
    weighted_sums = {}

    # Loop through each stock DataFrame and its corresponding weight
    for (tickers, df), weight in zip(stock_data_list.items(), weights_array):
        # Check available columns in the DataFrame
        available_columns = df.columns

        # Iterate through available columns and calculate weighted sum
        for col_type in available_columns:
            # Multiply the selected column by its corresponding weight
            weighted_col = df[col_type] * weight

            # Add to the sum
            if col_type not in weighted_sums:
                weighted_sums[col_type] = weighted_col
            else:
                weighted_sums[col_type] += weighted_col

    # Convert the weighted_sums dictionary to a DataFrame
    result_df = pd.DataFrame(weighted_sums)

    return result_df

In [ ]:
portfolio_data = portfolio(pc_vec, normalised_dict, 1)

In [ ]:
portfolio_data # check if this is correct
# must fix this
# gotta find what is causing  the NA

,Open_normalized,High_normalized,Low_normalized,Close_normalized,Volume,Returns
Date,,,,,,
2015-07-07 00:00:00-04:00,NaN,NaN,NaN,NaN,NaN,NaN
2015-07-08 00:00:00-04:00,NaN,NaN,NaN,NaN,NaN,NaN
2015-07-09 00:00:00-04:00,NaN,NaN,NaN,NaN,NaN,NaN
2015-07-10 00:00:00-04:00,NaN,NaN,NaN,NaN,NaN,NaN
2015-07-13 00:00:00-04:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2024-11-04 00:00:00-05:00,-137.946477,-138.133398,-141.409606,-137.724784,-1.472498e+08,0.024545
2024-11-05 00:00:00-05:00,-138.184634,-139.195724,-143.001928,-140.802131,-1.257121e+08,-0.062476
2024-11-06 00:00:00-05:00,-143.388153,-144.685166,-147.119202,-146.246983,-2.243988e+08,-0.182483


In [ ]:
portfolio_data.dropna(inplace=True)
print(portfolio_data)

                           Open_normalized  High_normalized  Low_normalized  \
Date                                                                          
2016-02-26 00:00:00-05:00        -5.519479        -5.518454       -5.569031   
2016-02-29 00:00:00-05:00        -5.480423        -5.527770       -5.575143   
2016-03-01 00:00:00-05:00        -5.516105        -5.605217       -5.605442   
2016-03-02 00:00:00-05:00        -5.618729        -5.642681       -5.705172   
2016-03-03 00:00:00-05:00        -5.652937        -5.682473       -5.713239   
...                                    ...              ...             ...   
2024-11-04 00:00:00-05:00      -137.946477      -138.133398     -141.409606   
2024-11-05 00:00:00-05:00      -138.184634      -139.195724     -143.001928   
2024-11-06 00:00:00-05:00      -143.388153      -144.685166     -147.119202   
2024-11-07 00:00:00-05:00      -146.355204      -147.454750     -151.330317   
2024-11-08 00:00:00-05:00      -148.622444      -148

# Indicators

In [ ]:
from ta import add_all_ta_features
from ta.utils import dropna

In [ ]:
def features_ta(weighted_data):

    weighted_data = add_all_ta_features(weighted_data, open="Open_normalized", high="High_normalized", low="Low_normalized", close="Close_normalized", volume="Volume", fillna=True)

    print(weighted_data)

    return weighted_data

In [ ]:
pc_feature_data = features_ta(portfolio_data)

/usr/local/lib/python3.10/dist-packages/ta/trend.py:1030: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  self._psar[i] = high2


                           Open_normalized  High_normalized  Low_normalized  \
Date                                                                          
2016-02-26 00:00:00-05:00        -5.519479        -5.518454       -5.569031   
2016-02-29 00:00:00-05:00        -5.480423        -5.527770       -5.575143   
2016-03-01 00:00:00-05:00        -5.516105        -5.605217       -5.605442   
2016-03-02 00:00:00-05:00        -5.618729        -5.642681       -5.705172   
2016-03-03 00:00:00-05:00        -5.652937        -5.682473       -5.713239   
...                                    ...              ...             ...   
2024-11-04 00:00:00-05:00      -137.946477      -138.133398     -141.409606   
2024-11-05 00:00:00-05:00      -138.184634      -139.195724     -143.001928   
2024-11-06 00:00:00-05:00      -143.388153      -144.685166     -147.119202   
2024-11-07 00:00:00-05:00      -146.355204      -147.454750     -151.330317   
2024-11-08 00:00:00-05:00      -148.622444      -148

/usr/local/lib/python3.10/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
pc_feature_data.to_csv('pc_feature_data.csv')

# Correlation of Indicators

we shift the indicators down by 1 as we want correlation between indicators at t-1 and returns at t

In [ ]:
def correlation_matrix(pc_data): # how many components were used

    # DataFrame to store all correlations
    all_correlations_df = pd.DataFrame()

    return_column = pc_data.iloc[:, 5]  # 5th column which contains all the returns
    indicator_columns = pc_data.iloc[:, (6):]  # All columns after the 6th

    # Shifting
    lagged_indicators = indicator_columns.shift(1)

    # Calculate the correlation
    correlation = lagged_indicators.corrwith(return_column)

    # Convert the correlation Series into a DataFrame
    correlation_df = correlation.to_frame(name='Correlation')

    return correlation_df.dropna()

In [ ]:
correlation_matrix_data = correlation_matrix(pc_feature_data)
print(correlation_matrix_data)

                     Correlation
volume_adi             -0.003397
volume_obv              0.019473
volume_cmf              0.008565
volume_fi               0.030053
volume_em              -0.043127
...                          ...
momentum_pvo_signal    -0.015324
momentum_pvo_hist      -0.016885
momentum_kama          -0.014729
others_dr               0.099904
others_cr               0.015950

[85 rows x 1 columns]


/usr/local/lib/python3.10/dist-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.10/dist-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


now to filter

In [ ]:
def correlation_filter(corr_mat, threshold):

    # Filter rows where the sum exceeds threshold value
    filtered_df = corr_mat[corr_mat['Correlation'].abs() >= threshold]

    # Extract the names of the rows that were kept
    included_rows = filtered_df.index.tolist()

    return filtered_df, included_rows

In [ ]:
filter_correlation_matrix, indicator_list = correlation_filter(correlation_matrix_data, 0.03)
print(filter_correlation_matrix)
print(indicator_list)

                  Correlation
volume_fi            0.030053
volume_em           -0.043127
volatility_bbhi     -0.047004
trend_mass_index     0.033242
trend_aroon_down     0.038599
momentum_rsi        -0.032202
others_dr            0.099904
['volume_fi', 'volume_em', 'volatility_bbhi', 'trend_mass_index', 'trend_aroon_down', 'momentum_rsi', 'others_dr']


In [ ]:
# Sort the rows based on the absolute values in decreasing order
df_sorted = filter_correlation_matrix.reindex(filter_correlation_matrix['Correlation'].abs().sort_values(ascending=False).index)

print(df_sorted)

                  Correlation
others_dr            0.099904
volatility_bbhi     -0.047004
volume_em           -0.043127
trend_aroon_down     0.038599
trend_mass_index     0.033242
momentum_rsi        -0.032202
volume_fi            0.030053


# Regression

In [ ]:
"""

From the top 10 correlated indicators
We will create a prediction model to predict the returns

"""



"""

A question arises whether it makes sense
To use others_dlr and others_dr as suitable indicators
as they themselves are designed for finding returns

"""


"""

And a final thing to wonder about
Is whether we should use the open, hight, low, etc. data
When predicting, since the indicators are based off of them

"""

'\n\nAnd a final thing to wonder about\nIs whether we should use the open, hight, low, etc. data\nWhen predicting, since the indicators are based off of them\n\n'

In [ ]:
# Extract the first 5 indicators from the index
first_5_names = df_sorted.index[:5]
print(first_5_names)

Index(['others_dr', 'volatility_bbhi', 'volume_em', 'trend_aroon_down',
       'trend_mass_index'],
      dtype='object')


In [ ]:
"""

Here we create a dataframe consisting of the
indicators we will build our model around
We will call it 'X_df' to resemble X and Y
Like how we'd generally do while performing regression

"""


# Use the existing time series as the index for the new DataFrame
X_df = pd.DataFrame(index=pc_feature_data.index, columns=first_5_names) # Changed from columns to index

# Let's populate each column with random data for now (replace this with your actual logic)
for name in first_5_names:
    X_df[name] = pc_feature_data[name]  # Changed from .loc to [] to access column data

# Display the new DataFrame
print(X_df)

                           others_dr  volatility_bbhi  volume_em  \
Date                                                               
2016-02-26 00:00:00-05:00   0.000000              0.0   0.000000   
2016-02-29 00:00:00-05:00  -0.343828              0.0   0.000159   
2016-03-01 00:00:00-05:00   2.900551              0.0   0.000005   
2016-03-02 00:00:00-05:00   0.517713              0.0   0.002180   
2016-03-03 00:00:00-05:00   0.292087              0.0   0.000337   
...                              ...              ...        ...   
2024-11-04 00:00:00-05:00   0.045828              0.0   1.189108   
2024-11-05 00:00:00-05:00   2.234418              0.0   4.018759   
2024-11-06 00:00:00-05:00   3.867024              0.0   5.210166   
2024-11-07 00:00:00-05:00   2.275700              0.0   7.477691   
2024-11-08 00:00:00-05:00  -0.487334              0.0   1.703454   

                           trend_aroon_down  trend_mass_index  
Date                                               

In [ ]:
"""

Doing the same for returns
and storing it as 'Y_df'

"""

Y_df = pc_feature_data['Returns']
print(Y_df)


Date
2016-02-26 00:00:00-05:00   -0.001464
2016-02-29 00:00:00-05:00    0.014866
2016-03-01 00:00:00-05:00   -0.141545
2016-03-02 00:00:00-05:00   -0.028411
2016-03-03 00:00:00-05:00   -0.021637
                               ...   
2024-11-04 00:00:00-05:00    0.024545
2024-11-05 00:00:00-05:00   -0.062476
2024-11-06 00:00:00-05:00   -0.182483
2024-11-07 00:00:00-05:00   -0.070544
2024-11-08 00:00:00-05:00   -0.018131
Name: Returns, Length: 2192, dtype: float64


In [ ]:
"""

Converting the above data to numpy arrays
So that we can carry out regression

"""

X = X_df.to_numpy()
Y = Y_df.to_numpy()

In [ ]:
""""
firslty,
we will manually conduct XGBOOST, LightGBM and Ridge

"""

!pip install xgboost
!pip install lightgbm

import xgboost as xgb
import lightgbm as lgb
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [ ]:
# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42) # gotta figure out the random_state aspect


In [ ]:
"""
XGBoost
"""

xgb_model = xgb.XGBRegressor()
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_train_pred = xgb_model.predict(X_train)
in_sample_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
in_sample_r2 = r2_score(y_train, y_train_pred)
print(f"In-sample RMSE: {in_sample_rmse}")
print(f"In-sample R²: {in_sample_r2}")

y_test_pred = xgb_model.predict(X_test)
out_of_sample_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
out_of_sample_r2 = r2_score(y_test, y_test_pred)
print(f"Out-of-sample RMSE: {out_of_sample_rmse}")
print(f"Out-of-sample R²: {out_of_sample_r2}")


In-sample RMSE: 0.0067735790635399685
In-sample R²: 0.9927710187155362
Out-of-sample RMSE: 0.02862299743544239
Out-of-sample R²: 0.8456083333430965


In [ ]:
"""
LightGBM
"""

lgb_model = lgb.LGBMRegressor()
lgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_train_pred = lgb_model.predict(X_train)
in_sample_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
in_sample_r2 = r2_score(y_train, y_train_pred)
print(f"In-sample RMSE: {in_sample_rmse}")
print(f"In-sample R²: {in_sample_r2}")

y_test_pred = lgb_model.predict(X_test)
out_of_sample_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
out_of_sample_r2 = r2_score(y_test, y_test_pred)
print(f"Out-of-sample RMSE: {out_of_sample_rmse}")
print(f"Out-of-sample R²: {out_of_sample_r2}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001014 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 793
[LightGBM] [Info] Number of data points in the train set: 1753, number of used features: 5
[LightGBM] [Info] Start training from score -0.004881
In-sample RMSE: 0.020861381832716047
In-sample R²: 0.931431154710336
Out-of-sample RMSE: 0.02939702721937992
Out-of-sample R²: 0.8371452383930087


In [ ]:
"""
RIDGE

"""

ridge_model = Ridge()
ridge_model.fit(X_train, y_train)

# Make predictions on the test set
y_train_pred = ridge_model.predict(X_train)
in_sample_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
in_sample_r2 = r2_score(y_train, y_train_pred)
print(f"In-sample RMSE: {in_sample_rmse}")
print(f"In-sample R²: {in_sample_r2}")

y_test_pred = ridge_model.predict(X_test)
out_of_sample_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
out_of_sample_r2 = r2_score(y_test, y_test_pred)
print(f"Out-of-sample RMSE: {out_of_sample_rmse}")
print(f"Out-of-sample R²: {out_of_sample_r2}")


In-sample RMSE: 0.031668626705714555
In-sample R²: 0.8419846952035329
Out-of-sample RMSE: 0.028690980468649416
Out-of-sample R²: 0.8448740651402415


In [ ]:
!pip install lazypredict

In [ ]:
from lazypredict.Supervised import LazyRegressor

In [ ]:
# Fit all regression models
reg = LazyRegressor(predictions=True)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)
models

100%|██████████| 42/42 [00:11<00:00,  3.54it/s]


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000198 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 795
[LightGBM] [Info] Number of data points in the train set: 1753, number of used features: 5
[LightGBM] [Info] Start training from score -0.004881


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
GradientBoostingRegressor,0.86,0.86,0.03,0.71
RandomForestRegressor,0.85,0.85,0.03,1.80
NuSVR,0.85,0.85,0.03,1.87
XGBRegressor,0.84,0.85,0.03,1.65
OrthogonalMatchingPursuit,0.84,0.85,0.03,0.08
Lars,0.84,0.84,0.03,0.02
TransformedTargetRegressor,0.84,0.84,0.03,0.04
LinearRegression,0.84,0.84,0.03,0.02
BayesianRidge,0.84,0.84,0.03,0.02
